# **Processamento Inicial de Dados da API do The Cancer Genome Atlas (TCGA)**
Desenvolvida pela Genomic Data Commons (GDC)

# Importação de Bibliotecas

In [1]:
import json

import pandas as pd
import requests

# Constantes

In [2]:
# URL HTML base da API TCGA / GDC
TCGA_API_ENDPOINT = 'https://api.gdc.cancer.gov'

# Endpoint para verificar status e versão da API
STATUS_ENDPOINT = f'{TCGA_API_ENDPOINT}/status'

# Endpoint de projetos relacionados aos programas
PROJECTS_ENDPOINT = f'{TCGA_API_ENDPOINT}/projects'

# Endpoint de casos relacionados aos projetos
CASES_ENDPOINT = f'{TCGA_API_ENDPOINT}/cases'

# Endpoint de arquivos relacionados aos casos
FILES_ENDPOINT = f'{TCGA_API_ENDPOINT}/files'

# Caminho da pasta de dados provisórios
INTERIM_DATA_PATH = '../../data/interim'

# Status da API

In [3]:
# Requisita o status e a versão da API ao endpoint
response = requests.get(STATUS_ENDPOINT)

# Imprime o conteúdo da resposta
print(response.content.decode('utf-8'))

{"commit":"3ed4235efe2f2d9c9969e1ce790e6b34a5c60c9d","data_release":"Data Release 41.0 - August 28, 2024","status":"OK","tag":"7.4.0","version":1}



# Projetos

## Todos os Projetos

In [4]:
# Requisita as informações contidas no endpoint
response = requests.get(f'{PROJECTS_ENDPOINT}/_mapping')

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))

# Armazena os campos contidos no endpoint
fields_projects = json.dumps(json_response['fields'])[1:-2]
fields_projects = fields_projects.replace('"', '')
fields_projects = fields_projects.replace(' ', '')

In [5]:
# Requisita o número padrão de objetos do endpoint
response = requests.post(
    url=PROJECTS_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json={'fields': 'name'}
)

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))

# Armazena o número total de projetos contidos no TCGA
total_projects = json.dumps(json_response['data']['pagination']['total'])
print(f'Total de projetos no TCGA: {total_projects}')

Total de projetos no TCGA: 86


In [6]:
# Parâmetros para a requisição ao endpoint
params = {
    'fields': fields_projects,
    'size': total_projects
}

# Requisita todos os objetos do endpoint
response = requests.post(
    url=PROJECTS_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json=params
)

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))['data']['hits']

# Transforma os valores booleanos do JSON em strings
for index in range(len(json_response)):
    obj = json_response[index]
    for key in obj:
        if isinstance(obj[key], bool): 
            obj[key] = str(obj[key])

# Cria um DataFrame pandas a partir do JSON dos projetos
df_all_projects = pd.json_normalize(json_response)

In [7]:
# Armazena o DataFrame em um arquivo CSV
file_name = 'tcga_all_projects.csv'
df_all_projects.to_csv(f'{INTERIM_DATA_PATH}/{file_name}', index=False)

# Imprime o DataFrame parcialmente
pd.set_option('display.max_colwidth', 100)
df_all_projects.head()

,id,primary_site,dbgap_accession_number,project_id,disease_type,name,releasable,state,released,summary.file_count,summary.data_categories,summary.experimental_strategies,summary.case_count,summary.file_size,program.dbgap_accession_number,program.program_id,program.name
0,TARGET-AML,"[Unknown, Hematopoietic and reticuloendothelial systems]",phs000465,TARGET-AML,"[Not Applicable, Myeloid Leukemias]",Acute Myeloid Leukemia,True,open,True,51339,"[{'file_count': 12095, 'case_count': 806, 'data_category': 'Simple Nucleotide Variation'}, {'fil...","[{'file_count': 28629, 'case_count': 2280, 'experimental_strategy': 'RNA-Seq'}, {'file_count': 5...",2492,104391798872383,phs000218,f1c391e9-8488-55a8-b777-302e786ea11d,TARGET
1,MATCH-Z1I,"[Bronchus and lung, Gallbladder, Pancreas, Unknown, Other and ill-defined sites in lip, oral cav...",phs002058,MATCH-Z1I,"[Squamous Cell Neoplasms, Epithelial Neoplasms, NOS, Nevi and Melanomas, Ductal and Lobular Neop...",Genomic Characterization CS-MATCH-0007 Arm Z1I,False,open,True,660,"[{'file_count': 327, 'case_count': 24, 'data_category': 'Simple Nucleotide Variation'}, {'file_c...","[{'file_count': 374, 'case_count': 24, 'experimental_strategy': 'WXS'}, {'file_count': 234, 'cas...",26,2396549656405,None,893b0820-b50b-5c75-85f5-3b028e251811,MATCH
2,HCMI-CMDC,"[Breast, Rectum, Nasal cavity and middle ear, Bronchus and lung, Other and unspecified parts of ...",None,HCMI-CMDC,"[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",NCI Cancer Model Development for the Human Cancer Model Initiative,True,open,True,20761,"[{'file_count': 8450, 'case_count': 278, 'data_category': 'Simple Nucleotide Variation'}, {'file...","[{'file_count': 8096, 'case_count': 276, 'experimental_strategy': 'WXS'}, {'file_count': 4333, '...",278,130722797290059,phs001486,a5448c11-d46a-56aa-a5e1-5c1aa06404df,HCMI
3,MATCH-W,"[Breast, Renal pelvis, Corpus uteri, Bladder, Rectum, Prostate gland, Liver and intrahepatic bil...",phs001948,MATCH-W,"[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",Genomic Characterization CS-MATCH-0007 Arm W,False,open,True,1091,"[{'file_count': 535, 'case_count': 44, 'data_category': 'Simple Nucleotide Variation'}, {'file_c...","[{'file_count': 614, 'case_count': 44, 'experimental_strategy': 'WXS'}, {'file_count': 387, 'cas...",45,3929920960515,None,893b0820-b50b-5c75-85f5-3b028e251811,MATCH
4,MATCH-Z1D,"[Breast, Corpus uteri, Bones, joints and articular cartilage of other and unspecified sites, Pro...",phs001859,MATCH-Z1D,"[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",Genomic Characterization CS-MATCH-0007 Arm Z1D,False,open,True,891,"[{'file_count': 440, 'case_count': 34, 'data_category': 'Simple Nucleotide Variation'}, {'file_c...","[{'file_count': 504, 'case_count': 34, 'experimental_strategy': 'WXS'}, {'file_count': 315, 'cas...",36,3282605560716,None,893b0820-b50b-5c75-85f5-3b028e251811,MATCH


## Projetos Lançados com miRNA-Seq e RNA-Seq

In [8]:
# Filtra os projetos lançados pelo TCGA e remove as colunas excedentes
df_projects = df_all_projects \
    .query('released == "True"') \
    .drop(
        columns=[
            'dbgap_accession_number',
            'id',
            'releasable',
            'released',
            'program.dbgap_accession_number',
            'program.name',
            'program.program_id',
            'state',
            'summary.data_categories',
            'summary.file_count',
            'summary.file_size'
        ]
    )

# Inicializa listas para a contagem de casos com miRNA-Seq e RNA-Seq nos projetos
miRNA_Seq_case_count = [0] * df_projects.shape[0]
RNA_Seq_case_count = [0] * df_projects.shape[0]

# Preenche listas com a contagem de casos com miRNA-Seq e RNA-Seq em cada projeto
for index in range(df_projects.shape[0]):
    for info in df_projects['summary.experimental_strategies'][index]:
        if info['experimental_strategy'] in ['miRNA-Seq', 'RNA-Seq']:
            if info['experimental_strategy'] == 'miRNA-Seq':
                miRNA_Seq_case_count[index] = info['case_count']
            else:
                RNA_Seq_case_count[index] = info['case_count']

# Adiciona a contagem de casos de miRNA-Seq e RNA-Seq ao DataFrame
df_projects['miRNA-Seq_case_count'] = miRNA_Seq_case_count
df_projects['RNA-Seq_case_count'] = RNA_Seq_case_count

# Filtra os projetos com miRNA-Seq e RNA-Seq como estratégia experimental
df_projects = df_projects \
    .query('(`miRNA-Seq_case_count` > 0) & (`RNA-Seq_case_count` > 0)') \
    .drop(columns='summary.experimental_strategies') \
    .reset_index(drop=True)

# Reorganiza e renomeia as colunas do DataFrame
columns = [
    'project_id',
    'name',
    'primary_site',
    'disease_type',
    'summary.case_count',
    'miRNA-Seq_case_count',
    'RNA-Seq_case_count'
]
df_projects = df_projects[columns] \
    .rename(
        columns={
            'name': 'project_name',
            'summary.case_count': 'case_count',
        }
    )

# Imprime o total de projetos com as características desejadas
print(f'Total de projetos de interesse: {df_projects.shape[0]}')

Total de projetos de interesse: 47


In [9]:
# Armazena o DataFrame em um arquivo CSV
file_name = 'tcga_projects_of_interest.csv'
df_projects.to_csv(f'{INTERIM_DATA_PATH}/{file_name}', index=False)

# Imprime o DataFrame parcialmente
pd.set_option('display.max_colwidth', 100)
df_projects.head()

,project_id,project_name,primary_site,disease_type,case_count,miRNA-Seq_case_count,RNA-Seq_case_count
0,TARGET-AML,Acute Myeloid Leukemia,"[Unknown, Hematopoietic and reticuloendothelial systems]","[Not Applicable, Myeloid Leukemias]",2492,1836,2280
1,HCMI-CMDC,NCI Cancer Model Development for the Human Cancer Model Initiative,"[Breast, Rectum, Nasal cavity and middle ear, Bronchus and lung, Other and unspecified parts of ...","[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",278,271,277
2,TCGA-PCPG,Pheochromocytoma and Paraganglioma,"[Other endocrine glands and related structures, Heart, mediastinum, and pleura, Connective, subc...",[Paragangliomas and Glomus Tumors],179,179,179
3,TCGA-THYM,Thymoma,"[Heart, mediastinum, and pleura, Thymus]",[Thymic Epithelial Neoplasms],124,124,120
4,TCGA-PAAD,Pancreatic Adenocarcinoma,[Pancreas],"[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",185,178,178


# Casos

## Casos com miRNA-Seq e RNA-Seq

In [10]:
# Campos de interesse para a requisição ao endpoint
fields = [
    'disease_type',
    'files.access',
    'files.data_category',
    'files.data_format',
    'files.data_type',
    'files.file_id',
    'files.experimental_strategy',
    'primary_site',
    'project.project_id'
]
fields = ','.join(fields)

# Inicializa o DataFrame de casos e arquivos
df_cases_and_files = pd.DataFrame()

# Busca pelos casos com miRNA-Seq e RNA-Seq de cada projeto de interesse
for index in range(df_projects.shape[0]):
    # Filtros empregados na requisição ao endpoint
    filters = {
        'op': 'and',
        'content': [
            {
                'op': '=',
                'content': {
                    'field': 'project.project_id',
                    'value': df_projects['project_id'][index]
                }
            },
            {
                'op': 'in',
                'content': {
                    'field': 'files.experimental_strategy',
                    'value': ['miRNA-Seq', 'RNA-Seq']
                }
            }
        ]
    }

    # Parâmetros para a requisição ao endpoint
    params = {
        'fields': fields,
        'filters': filters,
        'size': str(df_projects['case_count'][index])
    }

    # Requisita todos os objetos do projeto no endpoint
    response = requests.post(
        url=CASES_ENDPOINT,
        headers={'Content-Type': 'application/json'},
        json=params
    )

    # Converte o conteúdo da resposta para JSON
    json_response = json.loads(response.content.decode('utf-8'))

    # Converte o JSON para um DataFrame pandas
    df_project_cases = pd.json_normalize(json_response['data']['hits'])

    # Concatena os casos deste projeto com os restantes
    if df_cases_and_files.empty == False:
        df_cases_and_files = pd.concat(
            [df_cases_and_files, df_project_cases], ignore_index=True
        )
    else:
        df_cases_and_files = df_project_cases.copy()

# Renomeia algumas colunas do DataFrame
df_cases_and_files = df_cases_and_files.rename(
    columns={'id': 'case_id', 'project.project_id': 'project_id'}
)

In [11]:
# Cria o DataFrame de casos
df_cases = df_cases_and_files.drop(columns='files')

# Armazena o DataFrame em um arquivo CSV
file_name = 'tcga_cases_of_interest.csv'
df_cases.to_csv(f'{INTERIM_DATA_PATH}/{file_name}', index=False)

# Imprime o DataFrame parcialmente
df_cases

,case_id,primary_site,disease_type,project_id
0,58771370-5082-485e-ac68-13edfbd9ef0c,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
1,28da5b52-29ed-4817-a678-9206c5164da0,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
2,28dae019-4d93-42bf-9bb3-8e6e21bc1d29,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
3,28f7e68c-e0ab-4fb0-b835-2e506cb0012d,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
4,28ffdfb9-f051-4424-ad35-e6633e6bf42e,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
...,...,...,...,...
16606,a798a8cc-4e72-4a8c-9e20-74a14deafd12,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16607,fe2e89f7-8f4d-420a-a551-4877cf0fd1d3,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16608,db3a4986-55d5-4ecc-be73-59725dce3c33,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16609,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8,Corpus uteri,"Cystic, Mucinous and Serous Neoplasms",TCGA-UCEC


# Arquivos

## Arquivos com miRNA-Seq e RNA-Seq

In [12]:
# Explode as listas com dicionário sobre os arquivos dos casos
df_cases_and_files = df_cases_and_files.explode('files')

# Filtra os arquivos dos casos relacionados a miRNA-Seq e RNA-Seq
key = 'experimental_strategy'
df_cases_and_files = df_cases_and_files[
    df_cases_and_files['files'].apply(
        lambda x: (
            key in x and (x[key] == 'miRNA-Seq' or x[key] == 'RNA-Seq')
        )
    )
]

# Define o DataFrame de arquivos e casos
df_files = pd.concat(
    objs=[
        df_cases_and_files.reset_index(drop=True),
        pd.json_normalize(df_cases_and_files['files'])
    ],
    axis='columns'
)
df_files = df_files.drop(
    columns=['disease_type', 'files', 'primary_site', 'project_id']
)

In [13]:
# Armazena o DataFrame em um arquivo CSV
file_name = 'tcga_files_of_interest.csv'
df_files.to_csv(f'{INTERIM_DATA_PATH}/{file_name}', index=False)

# Imprime o DataFrame de arquivos e casos
df_files

,case_id,data_format,access,file_id,data_type,data_category,experimental_strategy
0,58771370-5082-485e-ac68-13edfbd9ef0c,TSV,controlled,7e37b56e-4a47-4e14-aa0c-2317e31f3e53,Transcript Fusion,Structural Variation,RNA-Seq
1,58771370-5082-485e-ac68-13edfbd9ef0c,BAM,controlled,2b213562-4c34-4976-9563-ef2e581eb993,Aligned Reads,Sequencing Reads,RNA-Seq
2,58771370-5082-485e-ac68-13edfbd9ef0c,BEDPE,controlled,4188fa7a-7b11-47ae-ad63-701f5a2ffecb,Transcript Fusion,Structural Variation,RNA-Seq
3,58771370-5082-485e-ac68-13edfbd9ef0c,BEDPE,controlled,dd250c15-0be7-4243-b147-d4c872587844,Transcript Fusion,Structural Variation,RNA-Seq
4,58771370-5082-485e-ac68-13edfbd9ef0c,BAM,controlled,619cea2a-d035-4979-ad83-b7863eefc337,Aligned Reads,Sequencing Reads,RNA-Seq
...,...,...,...,...,...,...,...
229791,43476c88-86bc-4a65-9327-f0d24a41c971,TXT,open,d78ca382-fc44-48fb-b40e-3c56bab0ba99,miRNA Expression Quantification,Transcriptome Profiling,miRNA-Seq
229792,43476c88-86bc-4a65-9327-f0d24a41c971,TSV,controlled,af52e052-4b18-41f6-8647-8370405c4b00,Splice Junction Quantification,Transcriptome Profiling,RNA-Seq
229793,43476c88-86bc-4a65-9327-f0d24a41c971,BAM,controlled,f491ad0e-78db-4ed0-9d80-9d26bb0f736d,Aligned Reads,Sequencing Reads,miRNA-Seq
229794,43476c88-86bc-4a65-9327-f0d24a41c971,TSV,open,36cd1c9b-ab47-4a09-96d6-848cd59c08dd,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq


## Detalhamento sobre as Amostras de um Caso

In [14]:
# Campos de interesse para a requisição ao endpoint
fields = [
    'access',
    'cases.samples.tissue_type',
    'cases.samples.tumor_descriptor',
    'cases.samples.sample_type',
    'created_datetime',
    'data_category',
    'data_format',
    'data_type',
    'experimental_strategy',
    'updated_datetime'
]
fields = ','.join(fields)

# Lista de UUID dos arquivos de interesse
file_ids = df_files \
    .query('case_id == "9ff6d022-6e23-4f44-a480-1b61929e6ee3"') \
    ['file_id'].to_list()

# Inicializa o DataFrame de arquivos
df_case_files = pd.DataFrame()

# Requisita as informações dos objetos desejados ao endpoint
for file_id in file_ids:
    # Filtro empregado na requisição ao endpoint
    filters = {
        'op': '=',
        'content': {
            'field': 'file_id',
            'value': file_id
        }
    }

    # Parâmetros para a requisição ao endpoint
    params = {
        'fields': fields,
        'filters': filters,
        'size': '1'
    }

    # Requisita todos os objetos do projeto no endpoint
    response = requests.post(
        url=FILES_ENDPOINT,
        headers={'Content-Type': 'application/json'},
        json=params
    )

    # Converte o conteúdo da resposta para JSON
    json_response = json.loads(response.content.decode('utf-8'))

    # Converte o JSON para um DataFrame pandas
    df_file_info = pd.json_normalize(json_response['data']['hits'])

    # Concatena os casos deste projeto com os restantes
    if df_case_files.empty == False:
        df_case_files = pd.concat([df_case_files, df_file_info], ignore_index=True)
    else:
        df_case_files = df_file_info.copy()

In [15]:
# Imprime os registros de interesse do DataFrame
pd.set_option('display.max_colwidth', 500)
df_case_files \
    .query('data_category == "Transcriptome Profiling"') \
    .sort_values(by=['data_type', 'created_datetime']) \
    .reset_index(drop=True) \
    [[
        'id',
        'data_type',
        'cases',
        'created_datetime',
        'updated_datetime'
    ]]

,id,data_type,cases,created_datetime,updated_datetime
0,b96247db-6f2a-4d26-9ac4-142e1c079e0e,Gene Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Primary', 'sample_type': 'Primary Tumor', 'tissue_type': 'Tumor'}]}]",2022-01-06T09:46:53.769541-06:00,2024-07-30T21:45:03.814480-05:00
1,93c8678d-7afd-4be2-92cb-c39ad7701b43,Gene Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Not Applicable', 'sample_type': 'Solid Tissue Normal', 'tissue_type': 'Normal'}]}]",2022-01-06T09:46:56.688571-06:00,2024-07-30T21:47:29.474323-05:00
2,d12e199e-8471-4f60-8da2-2b479db61ab4,Gene Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Primary', 'sample_type': 'Primary Tumor', 'tissue_type': 'Tumor'}]}]",2022-01-06T09:47:32.472999-06:00,2024-07-30T21:53:34.939697-05:00
3,167073fa-9e38-4f8d-af1f-301ed3a8b5f7,Gene Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Primary', 'sample_type': 'Primary Tumor', 'tissue_type': 'Tumor'}]}]",2022-01-06T09:47:37.123345-06:00,2024-07-30T21:41:07.815061-05:00
4,c54a604e-9379-46f1-938c-e2d09f8538d8,Gene Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Primary', 'sample_type': 'Primary Tumor', 'tissue_type': 'Tumor'}]}]",2022-01-06T09:47:57.280420-06:00,2024-07-30T21:43:27.830713-05:00
5,ceb4dd9b-6f10-4358-8312-3b126952d3cc,Gene Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Primary', 'sample_type': 'Primary Tumor', 'tissue_type': 'Tumor'}]}]",2022-01-06T09:55:35.726930-06:00,2024-07-30T21:41:18.605327-05:00
6,0cf6cded-942f-4141-a4a5-35afb7082f37,Isoform Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Not Applicable', 'sample_type': 'Solid Tissue Normal', 'tissue_type': 'Normal'}]}]",2019-10-10T11:23:13.777284-05:00,2024-07-29T13:21:12.651533-05:00
7,d39cc122-925b-4292-9fe1-cab1d031bbd7,Isoform Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Primary', 'sample_type': 'Primary Tumor', 'tissue_type': 'Tumor'}]}]",2019-12-13T08:15:27.709025-06:00,2024-07-29T10:48:30.938949-05:00
8,0afd73ac-8fbc-418f-9d58-b1da06da4c98,Isoform Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Primary', 'sample_type': 'Primary Tumor', 'tissue_type': 'Tumor'}]}]",2020-10-16T17:02:06.560539-05:00,2024-07-29T15:23:17.906969-05:00
9,c22a38cc-064b-46b8-a576-2fcbcfec7ceb,Isoform Expression Quantification,"[{'samples': [{'tumor_descriptor': 'Primary', 'sample_type': 'Primary Tumor', 'tissue_type': 'Tumor'}]}]",2020-10-16T17:06:28.600735-05:00,2024-07-29T15:24:06.021117-05:00
